# 📡 01 — Data Collection: Multi-Source Job Pipeline

**Ziel:** Stellenausschreibungen aus dem deutschen Data-Jobmarkt sammeln.

## 🎯 Problemstellung

Wer den Data-Jobmarkt analysieren will, hat ein Quellenproblem:
- **Eine Quelle reicht nicht.** Die Bundesagentur für Arbeit hat viele klassische Stellen, aber wenig Tech-Startups. LinkedIn/Indeed haben Tech-Jobs, aber kein offenes API.
- **Lösung:** Multi-Source-Pipeline mit zwei kostenlosen APIs, die sich ergänzen:
  - **Bundesagentur für Arbeit** (offizielle Stellen, unbegrenzt)
  - **Adzuna** (aggregiert LinkedIn, Indeed, StepStone — 250 Calls/Tag)

## 🏗️ Architektur

Das Pattern: **eine abstrakte Basisklasse, mehrere konkrete Quellen, einheitliches Output-Schema** (`UnifiedJob`). So bleibt der Cleaning-Code quellenagnostisch.

```
JobSource (abstract)
├── ArbeitsagenturSource
└── AdzunaSource
         ↓
    UnifiedJob (Schema)
         ↓
    pandas DataFrame
```

## 🚦 Demo-Modus

Dieses Notebook holt nur **50 Jobs zur Demonstration**. Der echte Sammel-Lauf läuft über `python -m src.collect_jobs` und sammelt 5.000+ Jobs.

In [ ]:
# Setup: Projekt-Root in sys.path damit src.* importierbar ist
import sys
from pathlib import Path

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 200)

## 1️⃣ Quelle 1 — Bundesagentur für Arbeit

Die **größte Stellendatenbank Deutschlands**, kostenlos zugänglich über die offizielle API.

**Endpoint:** `https://rest.arbeitsagentur.de/jobboerse/jobsuche-service/pc/v4/app/jobs`

**Trick:** Der API-Key `jobboerse-jobsuche` ist öffentlich (für die mobile App designed). Keine Registrierung nötig.

In [ ]:
from src.sources.arbeitsagentur import ArbeitsagenturSource

source = ArbeitsagenturSource()

# Kleine Demo-Sammlung: 50 'Data Analyst'-Stellen
jobs_aa = source.fetch(
    search_term="data analyst",
    max_results=50,
)

print(f"✅ Gesammelt: {len(jobs_aa)} Stellen")
print(f"\nBeispiel:\n  Titel:       {jobs_aa[0].job_title}\n  Arbeitgeber: {jobs_aa[0].employer_name}\n  Stadt:       {jobs_aa[0].job_city}\n  Source:      {jobs_aa[0].source}")

In [ ]:
# In DataFrame umwandeln für weitere Analyse
df_aa = pd.DataFrame([j.to_dict() for j in jobs_aa])
df_aa[["job_title", "employer_name", "job_city", "source"]].head(10)

## 2️⃣ Quelle 2 — Adzuna

Adzuna aggregiert über **LinkedIn, Indeed, StepStone, Monster** und Firmen-Websites. Stärker im Tech-/Startup-Bereich als die Arbeitsagentur.

**Endpoint:** `https://api.adzuna.com/v1/api/jobs/de/search/{page}`

**Limits:** 250 Calls/Tag im Free-Tier. Registrierung auf https://developer.adzuna.com erforderlich.

_Hinweis: Wenn `ADZUNA_APP_ID` nicht in der `.env` gesetzt ist, überspringt diese Zelle elegant._

In [ ]:
from src.sources.adzuna import AdzunaSource

adzuna = AdzunaSource()

if adzuna.app_id and adzuna.app_key:
    jobs_adz = adzuna.fetch(search_term="data analyst", max_results=50)
    df_adz = pd.DataFrame([j.to_dict() for j in jobs_adz])
    print(f"✅ Adzuna: {len(jobs_adz)} Stellen")
    display(df_adz[["job_title", "employer_name", "job_city", "source"]].head(10))
else:
    print("⚠️  Adzuna-Credentials nicht gesetzt — überspringe.")
    print("    Registrieren auf https://developer.adzuna.com und in .env eintragen:")
    print("    ADZUNA_APP_ID=...")
    print("    ADZUNA_APP_KEY=...")
    df_adz = pd.DataFrame()

## 3️⃣ Quellen-Vergleich

Wie unterschiedlich sind die beiden Quellen? Schauen wir uns an, welche **Arbeitgeber** sie liefern — das zeigt, ob sich die Quellen wirklich ergänzen oder doppeln.

In [ ]:
if not df_adz.empty:
    employers_aa = set(df_aa["employer_name"].dropna())
    employers_adz = set(df_adz["employer_name"].dropna())
    overlap = employers_aa & employers_adz

    print(f"Arbeitgeber bei Arbeitsagentur:  {len(employers_aa):>4}")
    print(f"Arbeitgeber bei Adzuna:          {len(employers_adz):>4}")
    print(f"Überschneidung:                  {len(overlap):>4}")
    print(f"Eindeutige Adzuna-Arbeitgeber:   {len(employers_adz - employers_aa):>4}")
    print()
    print("💡 Erkenntnis: Beide Quellen liefern überwiegend unterschiedliche\n"
          "   Arbeitgeber. Die Multi-Source-Strategie zahlt sich aus.")
else:
    print("Adzuna nicht verfügbar — Vergleich übersprungen.")

## 4️⃣ Zusammenführung & Deduplizierung

Beide Quellen liefern Daten im **selben Schema** (`UnifiedJob`), deshalb ist das Mergen trivial. Die `job_id` ist quellen-präfixiert (`arbeitsagentur_<refnr>`, `adzuna_<id>`), also auch bei Cross-Source-Deduplizierung eindeutig.

In [ ]:
# Quellen zusammenführen
all_jobs = pd.concat([df_aa, df_adz], ignore_index=True) if not df_adz.empty else df_aa.copy()

before = len(all_jobs)
all_jobs = all_jobs.drop_duplicates(subset="job_id")
after = len(all_jobs)

print(f"Vor Dedup:  {before:>4} Jobs")
print(f"Nach Dedup: {after:>4} Jobs ({before-after} Duplikate entfernt)")

# Verteilung pro Quelle
print("\nVerteilung:")
print(all_jobs["source"].value_counts().to_string())

## ✅ Zusammenfassung

**Was wir gebaut haben:**
- ✅ Abstrakte `JobSource`-Basisklasse mit einheitlichem `UnifiedJob`-Schema
- ✅ Zwei konkrete Implementierungen: Arbeitsagentur und Adzuna
- ✅ Multi-Source-Sammlung mit Deduplizierung
- ✅ Saubere Trennung: jede Quelle kapselt ihre eigene Pagination, Rate-Limiting, Field-Mapping

**Was als Nächstes passiert:**
→ Notebook **02** zeigt, wie die Roh-Daten bereinigt werden: Stadt-Extraktion, Salary-Parsing, Skill-Detection, Feature-Engineering.

**Production-Lauf:**
```bash
python -m src.collect_jobs   # ohne --max-results = alle verfügbaren Jobs
```

Liefert in der Praxis 5.000-7.000 Jobs deutschlandweit, die in `data/raw/jobs_raw_master.csv` gespeichert werden.